In [0]:
%run ../00_Setup/01_Config

In [0]:
%run ../Utils/Common_Utils

In [0]:
# Pipeline Configuration

PIPELINE_NAME = "Gold Above Average Customers"

SOURCE_TABLE = GOLD_CUSTOMER_360
TARGET_TABLE = GOLD_ABOVE_AVERAGE_CUSTOMERS

RUN_ID = generate_run_id()
START_TIME = datetime.now()

In [0]:
print("GOLD ABOVE AVERAGE CUSTOMERS PIPELINE")

print(f"Pipeline : {PIPELINE_NAME}")
print(f"Run ID : {RUN_ID}")
print(f"Target : {TARGET_TABLE}")

GOLD ABOVE AVERAGE CUSTOMERS PIPELINE
Pipeline : Gold Above Average Customers
Run ID : b9f7805b-3752-4617-aa19-e362fd9a4971
Target : retailmart.gold.above_average_customers


In [0]:
customer_360_df = spark.table(SOURCE_TABLE)
display(customer_360_df.limit(10))

customer_id,customer_city,customer_state,total_orders,total_items_purchased,total_spent,average_item_value,first_purchase,last_purchase,customer_lifetime_days
CUST_000018,Manaus,AM,7,21,26018.590000000004,1238.9804761904763,2021-04-02T21:19:00.000Z,2023-06-20T10:16:00.000Z,809
CUST_000036,Manaus,AM,3,8,10247.5,1280.9375,2021-03-02T03:32:00.000Z,2022-04-21T07:18:00.000Z,415
CUST_000066,Campo Grande,MS,4,4,4156.87,1039.2175,2021-06-09T23:00:00.000Z,2022-06-14T08:46:00.000Z,370
CUST_000089,Joao Pessoa,PB,5,15,21994.440000000002,1466.296,2021-04-10T06:34:00.000Z,2023-04-21T19:22:00.000Z,741
CUST_000095,Teresina,PI,1,1,1036.53,1036.53,2022-08-08T15:08:00.000Z,2022-08-08T15:08:00.000Z,0
CUST_000105,Recife,PE,4,6,11380.58,1896.7633333333333,2021-01-17T12:26:00.000Z,2022-07-30T07:39:00.000Z,559
CUST_000117,Manaus,AM,0,0,0.0,0.0,null,null,0
CUST_000119,Teresina,PI,8,11,13052.109999999999,1186.5554545454545,2021-02-01T12:21:00.000Z,2023-05-31T09:07:00.000Z,849
CUST_000136,Joao Pessoa,PB,1,3,3487.1,1162.3666666666666,2021-06-08T11:29:00.000Z,2021-06-08T11:29:00.000Z,0
CUST_000159,Porto Velho,RO,2,2,2271.27,1135.635,2021-11-11T15:22:00.000Z,2022-06-21T01:37:00.000Z,222


In [0]:
print(f"Total Customers : {customer_360_df.count()}")
customer_360_df.printSchema()

Total Customers : 15000
root
 |-- customer_id: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- total_orders: long (nullable = true)
 |-- total_items_purchased: long (nullable = true)
 |-- total_spent: double (nullable = true)
 |-- average_item_value: double (nullable = true)
 |-- first_purchase: timestamp (nullable = true)
 |-- last_purchase: timestamp (nullable = true)
 |-- customer_lifetime_days: integer (nullable = true)



In [0]:
spark.sql(f"""
SELECT

    COUNT(*) AS total_customers,
    AVG(total_spent) AS average_customer_spending,
    MIN(total_spent) AS minimum_spending,
    MAX(total_spent) AS maximum_spending

FROM {SOURCE_TABLE}

""").show()

+---------------+-------------------------+----------------+----------------+
|total_customers|average_customer_spending|minimum_spending|maximum_spending|
+---------------+-------------------------+----------------+----------------+
|          15000|        7486.081435999996|             0.0|        37171.46|
+---------------+-------------------------+----------------+----------------+



In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {TARGET_TABLE} AS
SELECT *
FROM {SOURCE_TABLE}
WHERE total_spent >
(
    SELECT AVG(total_spent)
    FROM {SOURCE_TABLE}
)
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [0]:
above_avg_df = spark.table(TARGET_TABLE)
display(above_avg_df.limit(10))

customer_id,customer_city,customer_state,total_orders,total_items_purchased,total_spent,average_item_value,first_purchase,last_purchase,customer_lifetime_days
CUST_000018,Manaus,AM,7,21,26018.590000000004,1238.9804761904763,2021-04-02T21:19:00.000Z,2023-06-20T10:16:00.000Z,809
CUST_000036,Manaus,AM,3,8,10247.5,1280.9375,2021-03-02T03:32:00.000Z,2022-04-21T07:18:00.000Z,415
CUST_000089,Joao Pessoa,PB,5,15,21994.440000000002,1466.296,2021-04-10T06:34:00.000Z,2023-04-21T19:22:00.000Z,741
CUST_000105,Recife,PE,4,6,11380.58,1896.7633333333333,2021-01-17T12:26:00.000Z,2022-07-30T07:39:00.000Z,559
CUST_000119,Teresina,PI,8,11,13052.109999999999,1186.5554545454545,2021-02-01T12:21:00.000Z,2023-05-31T09:07:00.000Z,849
CUST_000160,Teresina,PI,2,6,9413.55,1568.925,2021-05-26T05:53:00.000Z,2021-11-25T03:48:00.000Z,183
CUST_000188,Manaus,AM,4,7,8525.38,1217.9114285714284,2021-10-21T18:19:00.000Z,2022-12-17T15:54:00.000Z,422
CUST_000201,Florianopolis,SC,3,5,8224.21,1644.8419999999999,2022-12-28T23:13:00.000Z,2023-04-27T11:51:00.000Z,120
CUST_000205,Florianopolis,SC,5,5,8082.400000000001,1616.48,2021-10-29T18:56:00.000Z,2023-05-14T17:02:00.000Z,562
CUST_000227,Aracaju,SE,3,4,7539.76,1884.94,2021-12-18T00:24:00.000Z,2022-12-13T06:06:00.000Z,360


In [0]:
# Validation1
rows_written = above_avg_df.count()
print(f"Rows Written : {rows_written}")

Rows Written : 6488


In [0]:
# Validation2 
spark.sql(f"""
SELECT
COUNT(*) AS above_average_customers
FROM {TARGET_TABLE}
""").show()

+-----------------------+
|above_average_customers|
+-----------------------+
|                   6488|
+-----------------------+



In [0]:
# Validation3
spark.sql(f"""
SELECT

MIN(total_spent) AS minimum_above_average_spending

FROM {TARGET_TABLE}

""").show()

+------------------------------+
|minimum_above_average_spending|
+------------------------------+
|                       7486.12|
+------------------------------+



In [0]:
# Validation4
overall_avg = spark.sql(f"""
SELECT AVG(total_spent) AS avg_spent
FROM {SOURCE_TABLE}
""").collect()[0]["avg_spent"]

invalid_customers = above_avg_df.filter(
    f"total_spent <= {overall_avg}"
).count()

print(f"Invalid Customers : {invalid_customers}")

assert invalid_customers == 0

Invalid Customers : 0


In [0]:
source = SOURCE_TABLE
bronze_load_report(
    pipeline_name=PIPELINE_NAME,
    run_id=RUN_ID,
    source=source,
    target=TARGET_TABLE,
    rows_read=customer_360_df.count(),
    rows_written=rows_written,
    duplicate_count=0,
    start_time=START_TIME,
    status="SUCCESS"
)

LOAD REPORT
Pipeline        : Gold Above Average Customers
Run ID          : b9f7805b-3752-4617-aa19-e362fd9a4971
Source          : retailmart.gold.customer_360
Target          : retailmart.gold.above_average_customers
Rows Read       : 15000
Rows Written    : 6488
Duplicate Rows  : 0
Start Time      : 2026-07-19 10:31:11.465740
End Time        : 2026-07-19 10:31:32.670438
Duration (sec)  : 21.2
Status          : SUCCESS


## Engineering Observations

• A SQL subquery was used to dynamically calculate the average customer spending, eliminating the need for hardcoded threshold values.

• Customers whose total spending exceeded the overall average were identified as above-average customers, enabling targeted business analysis.

• The transformation reused the Gold Customer 360 table, avoiding repeated aggregations on transactional data and improving query efficiency.

• Business logic remains dynamic because the average spending is recalculated whenever the source data changes.

• The resulting Gold table can support loyalty programs, premium customer targeting, personalized marketing campaigns, and customer retention strategies.